# Phase 2: 10-Class MNIST Classification — Random Forest (From Scratch)
## Pipeline: Data Loading → Preprocessing → PCA → Training → Learning Curves → Evaluation

## 1. Setup & Imports

In [ ]:
import sys, os
sys.path.append('..')
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from ml_utils import train_test_split, StandardScaler
from ml_utils import classification_report, confusion_matrix
from ml_utils import compute_accuracy, k_fold_split, cross_validate
from ml_utils import plot_learning_curve, evaluate_model, show_misclassified
from sklearn.decomposition import PCA
from skimage.feature import hog
np.random.seed(42)
print("Libraries imported")

## 2. Configuration

In [ ]:
NUM_CLASSES = 10
TRAIN_SIZE, VAL_SIZE, TEST_SIZE = 0.70, 0.15, 0.15
PCA_VARIANCE = 0.95
K_FOLDS = 5
LC_FRACTIONS = [0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
RF_N_ESTIMATORS = 30
RF_MAX_DEPTH = 12
RF_MAX_FEATURES = 'sqrt'

## 3. Data Loading & Preprocessing

In [ ]:
X_train_raw = np.load('../../mnist.npz/x_train.npy')
y_train_raw = np.load('../../mnist.npz/y_train.npy')
X_test_raw  = np.load('../../mnist.npz/x_test.npy')
y_test_raw  = np.load('../../mnist.npz/y_test.npy')
X_all = np.concatenate([X_train_raw, X_test_raw], axis=0)
y_all = np.concatenate([y_train_raw, y_test_raw], axis=0)

min_count = min(np.bincount(y_all))
balanced_idx = []
for c in range(NUM_CLASSES):
    c_idx = np.where(y_all == c)[0]
    np.random.shuffle(c_idx)
    balanced_idx.extend(c_idx[:min_count])
balanced_idx = np.array(balanced_idx)
np.random.shuffle(balanced_idx)
X_balanced, y_balanced = X_all[balanced_idx], y_all[balanced_idx]
X_norm = X_balanced / 255.0

X_temp, X_test, y_temp, y_test = train_test_split(
    X_norm, y_balanced, test_size=TEST_SIZE/(TRAIN_SIZE+VAL_SIZE+TEST_SIZE),
    random_state=42, stratify=y_balanced)
val_adj = VAL_SIZE / (TRAIN_SIZE + VAL_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=val_adj, random_state=42, stratify=y_temp)
print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

### Visualise Dataset Samples
One example per class from the training set.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for c in range(NUM_CLASSES):
    r, col = divmod(c, 5)
    idx = np.where(y_train == c)[0][0]
    axes[r, col].imshow(X_train[idx], cmap='gray')
    axes[r, col].set_title(f'Digit {c}', fontweight='bold')
    axes[r, col].axis('off')
plt.suptitle('Training Set — One Sample per Class', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 4a. Flatten + Scaler + PCA Feature Extraction

In [ ]:
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_val_flat = X_val.reshape(X_val.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_flat)
X_val_sc = scaler.transform(X_val_flat)
X_test_sc = scaler.transform(X_test_flat)
pca = PCA(n_components=PCA_VARIANCE, svd_solver='full')
X_train_pca = pca.fit_transform(X_train_sc)
X_val_pca = pca.transform(X_val_sc)
X_test_pca = pca.transform(X_test_sc)
print(f"PCA: {X_train_sc.shape[1]} -> {X_train_pca.shape[1]} components ({np.sum(pca.explained_variance_ratio_):.4f} var)")

### Visualise PCA

In [ ]:
cum_var = np.cumsum(pca.explained_variance_ratio_)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(cum_var, linewidth=2)
axes[0].axhline(PCA_VARIANCE, ls='--', color='red', label=f'{PCA_VARIANCE:.0%} threshold')
axes[0].set_xlabel('# Components'); axes[0].set_ylabel('Cumulative Variance')
axes[0].set_title('PCA Explained Variance'); axes[0].legend(); axes[0].grid(alpha=.3)

recon = pca.inverse_transform(X_train_pca[0:1])
recon = scaler.inverse_transform(recon).reshape(28, 28)
axes[1].imshow(recon, cmap='gray')
axes[1].set_title(f'Reconstructed ({X_train_pca.shape[1]} PCs)'); axes[1].axis('off')
plt.suptitle('PCA Feature Extraction', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 4b. HOG Feature Extraction

**Histogram of Oriented Gradients** captures edge and gradient structure.
Applied to the 2D images (before flattening), then standardised.

In [ ]:
def extract_hog(images):
    feats = []
    for img in images:
        f = hog(img, orientations=9, pixels_per_cell=(4,4),
                cells_per_block=(2,2), block_norm='L2-Hys')
        feats.append(f)
    return np.array(feats)

X_train_hog = extract_hog(X_train)
X_val_hog   = extract_hog(X_val)
X_test_hog  = extract_hog(X_test)

scaler_hog = StandardScaler()
X_train_hog_sc = scaler_hog.fit_transform(X_train_hog)
X_val_hog_sc   = scaler_hog.transform(X_val_hog)
X_test_hog_sc  = scaler_hog.transform(X_test_hog)

print(f"HOG feature dim: {X_train_hog_sc.shape[1]}")
print("HOG features ready")

### Visualise HOG Features

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(X_train[0], cmap='gray')
axes[0].set_title('Original'); axes[0].axis('off')
_, hog_img = hog(X_train[0], orientations=9, pixels_per_cell=(4,4),
                 cells_per_block=(2,2), block_norm='L2-Hys', visualize=True)
axes[1].imshow(hog_img, cmap='hot')
axes[1].set_title('HOG Visualisation'); axes[1].axis('off')
plt.suptitle('HOG Feature Extraction', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 5. Decision Tree (From Scratch)

A recursive binary-split classifier using **Gini impurity**.
Supports random feature subsets at each node (used by Random Forest).

In [ ]:
class DecisionTreeClassifier:
    def __init__(self, max_depth=10, min_samples_split=4, max_features=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features  # int or None
        self.tree_ = None

    # ── Gini impurity ─────────────────────────────────────────────
    @staticmethod
    def _gini(y, n_classes):
        if len(y) == 0:
            return 0.0
        counts = np.bincount(y, minlength=n_classes)
        probs = counts / len(y)
        return 1.0 - np.sum(probs ** 2)

    # ── Find best split ──────────────────────────────────────────
    def _best_split(self, X, y, n_classes):
        n, d = X.shape
        if n < self.min_samples_split:
            return None, None

        parent_gini = self._gini(y, n_classes)
        best_gini = parent_gini
        best_feat, best_thresh = None, None

        # Random feature subset (for Random Forest)
        if self.max_features is not None:
            feats = np.random.choice(d, min(self.max_features, d), replace=False)
        else:
            feats = np.arange(d)

        for feat in feats:
            vals = X[:, feat]
            # Use percentile-based thresholds for speed
            thresholds = np.unique(np.percentile(vals, np.linspace(10, 90, 15)))
            for thresh in thresholds:
                left = y[vals <= thresh]
                right = y[vals > thresh]
                if len(left) == 0 or len(right) == 0:
                    continue
                g = (len(left) * self._gini(left, n_classes) +
                     len(right) * self._gini(right, n_classes)) / n
                if g < best_gini:
                    best_gini = g
                    best_feat = feat
                    best_thresh = thresh

        return best_feat, best_thresh

    # ── Build tree recursively ───────────────────────────────────
    def _build(self, X, y, depth, n_classes):
        majority = int(np.argmax(np.bincount(y, minlength=n_classes)))
        if depth >= self.max_depth or len(np.unique(y)) == 1 or len(y) < self.min_samples_split:
            return majority  # leaf

        feat, thresh = self._best_split(X, y, n_classes)
        if feat is None:
            return majority

        mask = X[:, feat] <= thresh
        left  = self._build(X[mask],  y[mask],  depth + 1, n_classes)
        right = self._build(X[~mask], y[~mask], depth + 1, n_classes)
        return (feat, thresh, left, right)

    def fit(self, X, y):
        self.n_classes_ = int(np.max(y) + 1)
        self.tree_ = self._build(X, y.astype(int), 0, self.n_classes_)
        return self

    def _traverse(self, x, node):
        if not isinstance(node, tuple):
            return node
        feat, thresh, left, right = node
        return self._traverse(x, left if x[feat] <= thresh else right)

    def predict(self, X):
        return np.array([self._traverse(x, self.tree_) for x in X])

## 6. Random Forest (From Scratch)

An ensemble of decision trees using **bootstrap sampling** (bagging)
and **random feature subsets** at each split. Final prediction by majority vote.

In [ ]:
class RandomForest:
    def __init__(self, n_estimators=30, max_depth=12, max_features='sqrt',
                 min_samples_split=4, random_state=42):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features
        self.min_samples_split = min_samples_split
        self.random_state = random_state
        self.trees = []

    def fit(self, X, y):
        rng = np.random.RandomState(self.random_state)
        n, d = X.shape
        self.classes_ = np.unique(y)

        if self.max_features == 'sqrt':
            mf = int(np.sqrt(d))
        elif isinstance(self.max_features, int):
            mf = self.max_features
        else:
            mf = d

        self.trees = []
        for i in range(self.n_estimators):
            # Bootstrap sample
            idx = rng.choice(n, n, replace=True)
            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=mf)
            tree.fit(X[idx], y[idx])
            self.trees.append(tree)
            if (i + 1) % 5 == 0 or i == 0:
                print(f"  Tree {i+1}/{self.n_estimators} built")
        return self

    def predict(self, X):
        all_preds = np.array([t.predict(X) for t in self.trees])  # (n_trees, n_samples)
        result = np.zeros(len(X), dtype=int)
        for i in range(len(X)):
            counts = np.bincount(all_preds[:, i].astype(int), minlength=len(self.classes_))
            result[i] = self.classes_[np.argmax(counts)]
        return result

## 7. Training (PCA Features)

In [ ]:
print("=" * 70)
print("TRAINING RANDOM FOREST (From Scratch)")
print("=" * 70)
rf = RandomForest(n_estimators=RF_N_ESTIMATORS, max_depth=RF_MAX_DEPTH,
                  max_features=RF_MAX_FEATURES, random_state=42)
rf.fit(X_train_pca, y_train)
rf_acc = evaluate_model(rf, X_test_pca, y_test, "Random Forest (From Scratch)")

In [ ]:
# Note: showing original images for misclassified PCA predictions
preds_pca = rf.predict(X_test_pca)
show_misclassified(y_test, preds_pca, X_test,
                   title='Misclassified — PCA Features')

### Random Forest — Flatten Features

In [ ]:
print("\n" + "="*70)
print("Random Forest — Flatten Features")
print("="*70)
model_flatten = RandomForest(n_estimators=RF_N_ESTIMATORS, max_depth=RF_MAX_DEPTH, max_features=RF_MAX_FEATURES, random_state=42)
model_flatten.fit(X_train_sc, y_train)
evaluate_model(model_flatten, X_test_sc, y_test, "Random Forest (Flatten)")

In [ ]:
# Note: showing original images for misclassified Flatten predictions
preds_flatten = model_flatten.predict(X_test_sc)
show_misclassified(y_test, preds_flatten, X_test,
                   title='Misclassified — Flatten Features')

### Random Forest — HOG Features

In [ ]:
print("\n" + "="*70)
print("Random Forest — HOG Features")
print("="*70)
model_hog = RandomForest(n_estimators=RF_N_ESTIMATORS, max_depth=RF_MAX_DEPTH, max_features=RF_MAX_FEATURES, random_state=42)
model_hog.fit(X_train_hog_sc, y_train)
evaluate_model(model_hog, X_test_hog_sc, y_test, "Random Forest (HOG)")

In [ ]:
# Note: showing original images for misclassified HOG predictions
preds_hog = model_hog.predict(X_test_hog_sc)
show_misclassified(y_test, preds_hog, X_test,
                   title='Misclassified — HOG Features')

## 8. Learning Curve

In [ ]:
plot_learning_curve(RandomForest,
                    {'n_estimators': 15, 'max_depth': RF_MAX_DEPTH,
                     'max_features': RF_MAX_FEATURES, 'random_state': 42},
                    X_train_pca, y_train, X_val_pca, y_val, LC_FRACTIONS,
                    "Learning Curve - Random Forest (From Scratch)")

## 9. Summary

**Analysis**: Random Forest uses bagging (bootstrap aggregation) to reduce
variance. Each tree sees a random subset of data and features, making the
ensemble more robust than any individual tree. The gap between training and
validation curves indicates overfitting level. `max_depth` and `n_estimators`
control the bias-variance tradeoff.